# 01 · Load and validate the data

**Steps:** download the raw CSV → create the MySQL table → load → run data quality checks → create the clean view.

The raw table is kept exactly as received; all cleaning happens in the `email_test_clean` view (`sql/03_clean_view.sql`).

In [1]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

sys.path.append(str(Path.cwd().parent / "src"))
import ab_test as ab

pd.set_option("display.float_format", "{:,.4f}".format)
ab.apply_chart_style()

## 1. Download the raw data

In [2]:
DATA_URL = "https://hillstorm1.s3.us-east-2.amazonaws.com/hillstorm_no_indices.csv.gz"
RAW_FILE = ab.REPO_ROOT / "data" / "hillstrom_email_test.csv"

if RAW_FILE.exists():
    raw = pd.read_csv(RAW_FILE)
else:
    raw = pd.read_csv(DATA_URL)
    raw.to_csv(RAW_FILE, index=False)

print(raw.shape)
raw.head()

(64000, 12)


,recency,history_segment,history,mens,womens,zip_code,newbie,channel,segment,visit,conversion,spend
0,10,2) $100 - $200,142.4400,1,0,Surburban,0,Phone,Womens E-Mail,0,0,0.0000
1,6,3) $200 - $350,329.0800,1,1,Rural,1,Web,No E-Mail,0,0,0.0000
2,7,2) $100 - $200,180.6500,0,1,Surburban,1,Web,Womens E-Mail,0,0,0.0000
3,9,5) $500 - $750,675.8300,1,0,Rural,1,Web,Mens E-Mail,0,0,0.0000
4,2,1) $0 - $100,45.3400,1,0,Urban,0,Web,Womens E-Mail,0,0,0.0000


**Observations before loading**
- 64,000 rows, 12 columns, as documented by the source.
- `zip_code` contains the source typo **"Surburban"**. It is fixed in the clean view, not in the raw table.
- There is no customer ID, so one is created (1 to 64,000) to give every row a stable key.

In [3]:
raw.insert(0, "customer_id", range(1, len(raw) + 1))

## 2. Create the table and load

In [4]:
server = ab.get_engine(database=None)
ab.run_sql_file(server, ab.REPO_ROOT / "sql" / "01_create_table.sql")

engine = ab.get_engine()
with engine.begin() as conn:
    conn.exec_driver_sql("TRUNCATE TABLE email_test_raw")      # makes the load safe to re-run
raw.to_sql("email_test_raw", engine, if_exists="append", index=False, chunksize=5000)

pd.read_sql("SELECT COUNT(*) AS rows_loaded FROM email_test_raw", engine)

MySQL password for root:  ········


,rows_loaded
0,64000


## 3. Data quality checks

In [5]:
rule_checks, allowed_values, history_bins = ab.run_sql_file(
    engine, ab.REPO_ROOT / "sql" / "02_data_quality_checks.sql")
rule_checks

,check_name,failed_rows
0,duplicate customer_id,0
1,recency outside 1-12,0
2,history <= 0,0
3,spend < 0,0
4,flag not 0/1,0
5,converted without visiting,0
6,spend > 0 but not converted,0
7,converted but spend = 0,0
8,bought neither mens nor womens,0


All 9 rules return 0 failed rows: no duplicates, valid ranges, and the purchase funnel is consistent (no purchase without a visit, no spend without a purchase, every purchase has spend). That last point means **revenue per customer = conversion rate × spend per buyer** holds exactly in this data.

In [6]:
allowed_values

,column_name,value,customers
0,channel,Multichannel,7762
1,channel,Phone,28021
2,channel,Web,28217
3,history_segment,1) $0 - $100,22970
4,history_segment,2) $100 - $200,14254
5,history_segment,3) $200 - $350,12289
6,history_segment,4) $350 - $500,6409
7,history_segment,5) $500 - $750,4911
8,history_segment,"6) $750 - $1,000",1859
9,history_segment,"7) $1,000 +",1308


In [7]:
history_bins

,history_segment,min_history,max_history,customers
0,1) $0 - $100,29.9900,99.9900,22970
1,2) $100 - $200,100.0000,199.9800,14254
2,3) $200 - $350,200.0000,349.9600,12289
3,4) $350 - $500,350.0100,499.9500,6409
4,5) $500 - $750,500.0000,749.7900,4911
5,"6) $750 - $1,000",750.0100,999.7500,1859
6,"7) $1,000 +","1,000.1500","3,345.9300",1308


- Group sizes are close to equal thirds; whether the small differences are random is tested in notebook 02.
- History bins match the amounts. The lowest history is $29.99, not $0: every customer spent at least that much, which looks like a minimum order value rather than an error.
- Spend history is right-skewed: 36% of customers are in the lowest bin and 2% spent $1,000+.

## 4. Clean view

In [8]:
ab.run_sql_file(engine, ab.REPO_ROOT / "sql" / "03_clean_view.sql")
df = ab.load_clean(engine)
print(df.shape)
display(df["area"].value_counts().to_frame())
df["past_purchase"].value_counts().to_frame()

(64000, 13)


,count
area,
Suburban,28776
Urban,25661
Rural,9563


,count
past_purchase,
Mens only,28818
Womens only,28734
Both,6448


The view has all 64,000 rows, the area typo is fixed, and the pre-specified segments (recency band, past purchase, new vs existing) exist as columns.